In [0]:
#Create a Spark session object and name it Basic_Stats.
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Basic_Stats")
    .getOrCreate()
)

In [0]:
#Upload to Databricks the data frame corr_data.csv available in Canvas. Make sure to set the first row to column header.  
file_path = "/Workspace/Users/jessica.vargas006@mymdc.net/corr_data.csv"

df = (
    spark.read
         .option("header", "true")      
         .option("inferSchema", "true") 
         .csv(file_path)
)

df.printSchema()
df.show(5)


root
 |-- YearsExperience: double (nullable = true)
 |-- Salary: integer (nullable = true)

+---------------+------+
|YearsExperience|Salary|
+---------------+------+
|            1.1| 39343|
|            1.3| 46205|
|            1.5| 37731|
|            2.0| 43525|
|            2.2| 39891|
+---------------+------+
only showing top 5 rows


In [0]:
#Count the number of records in this dataframe. 
record_count = df.count()
print("Number of records in the dataframe:", record_count)


Number of records in the dataframe: 30


In [0]:
#Build a basic visualization with the following characteristics(hint: Run first df1.display() ):
#a.Scatter Plot
#c.YearsExperience and Salary included in the values
#d.Include a smooth line and explain your thoughts about the trend

df1 = df.select("YearsExperience", "Salary")
display(df1)

#The scatter plot shows a clear upward trend: as YearsExperience increases, Salary also increases. The smooth line is almost straight and increasing, which suggests a strong positive, roughly linear relationship between years of experience and salary.


YearsExperience,Salary
1.1,39343
1.3,46205
1.5,37731
2.0,43525
2.2,39891
2.9,56642
3.0,60150
3.2,54445
3.2,64445
3.7,57189


Databricks visualization. Run in Databricks to view.

In [0]:
#Calculate the Pearson correlation coefficient and explain your thoughts about it.

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

# Create "features" vector with both columns
assembler = VectorAssembler(
    inputCols=["YearsExperience", "Salary"],
    outputCol="features"
)

df_new = assembler.transform(df).select("features")
df_new.show(5, truncate=False)


+-------------+
|features     |
+-------------+
|[1.1,39343.0]|
|[1.3,46205.0]|
|[1.5,37731.0]|
|[2.0,43525.0]|
|[2.2,39891.0]|
+-------------+
only showing top 5 rows


In [0]:
pearson_corr_df = Correlation.corr(df_new, 'features', 'pearson')
pearson_corr_df.show(truncate=False)

#The Pearson correlation coefficient between YearsExperience and Salary is approximately 0.94. This indicates a very strong positive linear relationship: as years of experience increase, salary tends to increase in an almost linear way. The points in the scatter plot align closely around an upward-sloping line.

+----------------------------------------------------------------------------------+
|pearson(features)                                                                 |
+----------------------------------------------------------------------------------+
|1.0                 0.9384304417291862  \n0.9384304417291862  1.0                 |
+----------------------------------------------------------------------------------+



In [0]:
spearman_corr_df = Correlation.corr(df_new, 'features', 'spearman')
spearman_corr_df.show(truncate=False)

#The Spearman correlation coefficient between YearsExperience and Salary is approximately 0.96. This shows a very strong positive monotonic relationship: as years of experience go up, the ranking of salaries also consistently increases. The fact that Spearman is slightly higher than Pearson suggests the relationship is not only linear but also strongly monotonic (higher experience → higher salary almost always).

**Pearson correlation:**

- Measures linear relationship between two numeric variables.
- Uses the actual values.
- Sensitive to outliers and assumes the relationship is roughly linear.

**Spearman correlation:**

- Measures monotonic relationship (whether variables move in the same order), not necessarily linear.
- Uses the ranks of the data rather than their raw values.
- Less sensitive to outliers and can capture non-linear but still consistently increasing/decreasing relationships.

**For this dataset:**

Both Pearson ~0.94 and Spearman ~0.96 are very high and similar, which suggests the relationship between years of experience and salary is strong, positive, and close to linear.